# Dataset Generation Notebook

This notebook generates an Excel file containing crystallographic dataset metadata.

## Output File Structure

The Excel file contains the following columns:

- `pg_number` – Plane group number
- `seed` – Random seed used for generation
- `wyckoff_letters` – Wyckoff position labels
- `image_size` – Dimensions of the generated image
- `labels` – Classification labels for the structure

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import os

In [2]:
from mtflearn.utils import normalize_image
from symmlearn.lattice import (
    PlaneGroup,
    PGLattice,
    PGImage,
    WyckoffStructure
)
from symmlearn.utils import check_random_generator

In [2]:
def load_structures():
    current_folder = os.getcwd()
    parent = os.path.dirname(current_folder)
    structure_data = np.load(parent+"\\data\\structure data\\structure_data.pkl", allow_pickle=True)
    return structure_data

# load structure dictionary from step 1 (notebook 1)
# structure_data is a python dictionary
structure_data = load_structures()

# structure data's keys are plane group numbers from 1 to 17
structure_data.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17])

In [3]:
# give a pg_number and letters, 

def get_typical_structures_in_excel(pg_num, structures, elements = ['C', 'B'], sample_counts = 500, stop_counts = 20, image_size = 512, seed=None):
    """
    Filter typical structures for each structure type
    
    Args:
        pg_num: Plane group number
        structures: List of structure letters
        elements: List of elements
        sample_counts: Total target sample count
        stop_counts: Maximum attempts per structure
        image_size: Image size
    
    Returns:
        List of selected structure parameters [pg_num, seed, structure, image_size, label]
    """
    # Calculate the number of typical structures to find for each structure
    main_rng = check_random_generator(seed)
    num_structures = len(structures)
    target_per_structure = max(1, round(sample_counts / num_structures))
    stop_counts = stop_counts*target_per_structure

    selected_results = []
    
    # Use tqdm to show progress
    pbar = tqdm(structures, desc=f"PG{pg_num} - Processing")
    
    for structure in pbar:
        selected_for_structure = []
        attempts = 0
        
        # Update progress bar description to show current structure
        pbar.set_description(f"PG{pg_num} - Structure: {structure}")

        while len(selected_for_structure) < target_per_structure and attempts < stop_counts:
            # Generate random seed
            seed = main_rng.random.randint(0, 2**31)
            rng = check_random_generator(seed)
            
            try:
                # Create Wyckoff structure
                wy = WyckoffStructure(pg_num=pg_num, structure_letters=structure)
                struct_dict = wy.to_structure_dict(seed=rng)
                
                # Generate lattice
                pg = PlaneGroup(plane_group=pg_num)
                lattice = pg.generate_lattice(struct_dict, size=image_size, seed=rng, max_samples=5, angle_deg=0, sigma_method='min',verbose=False)
                
                # Convert lattice to image
                pgi = lattice.get_image(image_size=image_size, sigma_map=None, seed=rng)
                pgi.compute_symm_maps(patch_size=None, n_max=20, return_angle=True, crop=False)
                
                # Check if it is a typical structure
                if pgi.is_typical():
                    label = pgi.pg_number # this is updated pg number
                    selected_for_structure.append([pg_num, seed, structure, image_size, label])
                    
            except Exception:
                # Continue trying if an error occurs
                pass
                
            attempts += 1
            # Update the number of attempts for the current structure after each attempt
            pbar.set_postfix_str(f"Attempts: {attempts}/{stop_counts}")
        
        # Record results for current structure
        selected_results.extend(selected_for_structure)
    
    return selected_results

In [7]:
def write_to_excel(data, filename='seeds.xlsx'):
    current_folder = os.getcwd()
    parent = os.path.dirname(current_folder)
    data_folder = os.path.join(parent, "data", "accumulated seeds")

    os.makedirs(data_folder, exist_ok=True)
    full_path = os.path.join(data_folder, filename)
    df_new = pd.DataFrame(data)
    
    if os.path.exists(full_path):
        existing_df = pd.read_excel(full_path, header=None)
        combined_df = pd.concat([existing_df, df_new], ignore_index=True)
        combined_df.to_excel(full_path, index=False, header=False)
        print(f"data has been added into {filename}")
    else:
        df_new.to_excel(full_path, index=False, header=False)
        print(f"Create new file, data has been written into {full_path}")

In [ ]:
# master seed 

seed = 48
rng = check_random_generator(seed)

## PG number 1

In [8]:
pg_num = 1
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_1 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG1 - Structure: aaaaaaaaaaaaaaaaaaaa: 100%|█████████████████████████| 20/20 [03:59<00:00, 11.98s/it, Attempts: 10/200]


In [9]:
write_to_excel(accumulated_seeds_1, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 2

In [10]:
pg_num = 2
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_2 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG2 - Structure: abcdeeeeeeee: 100%|███████████████████████████████| 155/155 [30:00<00:00, 11.62s/it, Attempts: 10/200]


In [11]:
write_to_excel(accumulated_seeds_2, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 3

In [12]:
pg_num = 3
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_3 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG3 - Structure: bbbbbbbbbbbb: 100%|█████████████████████████████| 251/251 [1:06:37<00:00, 15.93s/it, Attempts: 38/200]


In [13]:
write_to_excel(accumulated_seeds_3, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG Number 4

In [14]:
pg_num = 4
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_4 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG4 - Structure: aaaaaaaaaa: 100%|███████████████████████████████████| 10/10 [02:02<00:00, 12.24s/it, Attempts: 12/200]


In [15]:
write_to_excel(accumulated_seeds_4, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG Number 5

In [16]:
pg_num = 5
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_5 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG5 - Structure: aaaaaaaaaa: 100%|███████████████████████████████████| 35/35 [12:04<00:00, 20.71s/it, Attempts: 35/200]


In [17]:
write_to_excel(accumulated_seeds_5, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 6

In [18]:
pg_num = 6
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_6 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG6 - Structure: abcdhh: 100%|███████████████████████████████████| 565/565 [1:39:04<00:00, 10.52s/it, Attempts: 17/200]


In [19]:
write_to_excel(accumulated_seeds_6, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 7

In [20]:
pg_num = 7
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_7 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG7 - Structure: abcccccccc: 100%|█████████████████████████████████| 120/120 [46:17<00:00, 23.15s/it, Attempts: 16/200]


In [21]:
write_to_excel(accumulated_seeds_7, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 8

In [22]:
pg_num = 8
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_8 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG8 - Structure: abcccc: 100%|███████████████████████████████████████| 20/20 [05:59<00:00, 17.97s/it, Attempts: 15/200]


In [23]:
write_to_excel(accumulated_seeds_8, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG Number 9

In [24]:
pg_num = 9
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_9 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG9 - Structure: abceee: 100%|█████████████████████████████████████| 160/160 [52:27<00:00, 19.67s/it, Attempts: 19/200]


In [25]:
write_to_excel(accumulated_seeds_9, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 10

In [21]:
pg_num = 10
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_10 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG10 - Structure: abcdddd: 100%|█████████████████████████████████████| 40/40 [12:38<00:00, 18.96s/it, Attempts: 20/200]


In [53]:
write_to_excel(accumulated_seeds_10, filename='seeds.xlsx')

Create new file, data has been written into D:\work\symmlearn\data\accumulated seeds\seeds.xlsx


## PG number 11

In [38]:
pg_num = 11
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_11 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG11 - Structure: abcfff: 100%|████████████████████████████████████| 257/257 [52:28<00:00, 12.25s/it, Attempts: 11/200]


In [54]:
write_to_excel(accumulated_seeds_11, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 12

In [36]:
pg_num = 12
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_12 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG12 - Structure: abcccc: 100%|██████████████████████████████████████| 38/38 [15:02<00:00, 23.75s/it, Attempts: 35/200]


In [55]:
write_to_excel(accumulated_seeds_12, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 13

In [40]:
pg_num = 13
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_13 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG13 - Structure: abcddddd: 100%|████████████████████████████████████| 54/54 [23:39<00:00, 26.29s/it, Attempts: 31/200]


In [56]:
write_to_excel(accumulated_seeds_13, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 14

In [42]:
pg_num = 14
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_14 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG14 - Structure: abcddddd: 100%|██████████████████████████████████| 123/123 [43:18<00:00, 21.13s/it, Attempts: 22/200]


In [57]:
write_to_excel(accumulated_seeds_14, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 15

In [44]:
pg_num = 15
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_15 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG15 - Structure: abccccc: 100%|█████████████████████████████████████| 59/59 [30:31<00:00, 31.04s/it, Attempts: 22/200]


In [58]:
write_to_excel(accumulated_seeds_15, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 16

In [59]:
pg_num = 16
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_16 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG16 - Structure: abcdd: 100%|███████████████████████████████████████| 26/26 [20:56<00:00, 48.34s/it, Attempts: 70/200]


In [60]:
write_to_excel(accumulated_seeds_16, filename='seeds.xlsx')

data has been added into seeds.xlsx


## PG number 17

In [63]:
pg_num = 17
counts_per_structure = 10
structures = structure_data[pg_num]
sample_counts = len(structures)*counts_per_structure
accumulated_seeds_17 = get_typical_structures_in_excel(pg_num=pg_num, structures=structures, sample_counts=sample_counts, seed=rng)

PG17 - Structure: abcee: 100%|███████████████████████████████████████| 73/73 [30:47<00:00, 25.30s/it, Attempts: 14/200]


In [64]:
write_to_excel(accumulated_seeds_17, filename='seeds.xlsx')

data has been added into seeds.xlsx
